In [1]:
import polars as pl
import pandas as pd
import gc
from features import apply_type_casting, generate_features, cat_features
from catboost import CatBoostClassifier

In [2]:
df_train = pl.scan_parquet('../data/train_full.parquet').drop('target')
df_pretest = apply_type_casting(pl.scan_parquet('../data/pretest.parquet'))
df_test = apply_type_casting(pl.scan_parquet('../data/test.parquet'))

all_preds = []
all_event_ids = []

model = CatBoostClassifier()
model.load_model('../models/catboost_v4.cbm')

N_CHUNKS = 10

for i in range(N_CHUNKS):
    print(f'Обработка чанка {i+1}/{N_CHUNKS}')
    batch_train = df_train.filter((pl.col('customer_id') % N_CHUNKS) == i)
    batch_pretest = df_pretest.filter((pl.col('customer_id') % N_CHUNKS) == i)
    batch_test = df_test.filter((pl.col('customer_id') % N_CHUNKS) == i)
    
    batch = pl.concat([batch_train, batch_pretest, batch_test]).unique(subset=['event_id'], keep='last')
    batch = generate_features(batch)
    
    batch = batch.join(batch_test.select('event_id'), on='event_id').collect().to_pandas()
    
    event_ids = batch['event_id'].values
    X_test = batch.drop(['customer_id', 'event_id', 'event_dttm', 'event_date'], axis=1)
    
    preds = model.predict_proba(X_test)[:, 1]
    
    all_preds.extend(preds)
    all_event_ids.extend(event_ids)
    
    del batch, X_test, preds, event_ids
    gc.collect()

Обработка чанка 1/10
Обработка чанка 2/10
Обработка чанка 3/10
Обработка чанка 4/10
Обработка чанка 5/10
Обработка чанка 6/10
Обработка чанка 7/10
Обработка чанка 8/10
Обработка чанка 9/10
Обработка чанка 10/10


In [3]:
df_submit = pd.DataFrame({
    'event_id': all_event_ids,
    'predict': all_preds
})

df_submit.to_csv('../data/submission_v4_4.csv', index=False)